In [2]:
"""
9-Ball Billiards Foul Detector — Without ByteTrack (cause it dont work)
=============================================================================
"""

import os
import time
import threading
import dataclasses
from datetime import datetime
from typing import Optional, List
from collections import deque, Counter, defaultdict

import cv2
import tkinter as tk
from tkinter import filedialog, messagebox
from PIL import Image, ImageTk
from ultralytics import YOLO


# ═══════════════════════════ Configuration ════════════════════════════════
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
DEFAULT_MODEL_PATH = os.path.join(
    "..", "Trained_Models", "YOLO11n", "weights", "best.pt"
)

CLASS_NAMES = {
    0: "1-ball", 1: "2-ball", 2: "3-ball", 3: "4-ball", 4: "5-ball",
    5: "6-ball", 6: "7-ball", 7: "8-ball", 8: "9-ball",
    9: "cue",   10: "cue_stick", 11: "pocket", 12: "rack",
}
BALL_CLASSES = set(range(9))     # 0–8  numbered object balls
CUE_CLASS    = 9
POCKET_CLASS = 11
RACK_CLASS   = 12

# Detection
CONF_THRESHOLD = 0.35

# Shot / velocity  (pixels-per-frame, calibrated for ~720p @ 30 fps)
SHOT_START_VEL = 8.0    # cue faster than this → shot has started
SHOT_END_VEL   = 5.0    # cue slower than this counts toward "at rest"

# Timing (all in seconds)
SHOT_END_SECS     = 1.0   # cue at rest this long → shot is over
SCRATCH_REST_SECS = 1.0   # cue resting inside pocket bbox → scratch foul
CUE_MISSING_SECS       = 12.0  # cue undetected this long → foul
POCKET_DISAPPEAR_SECS  =  0.5  # cue seen near pocket then missing → foul
FOUL_BANNER_SECS  = 3.0   # on-screen foul banner duration

# Contact confirmation
MOVEMENT_CONFIRM_FRAMES = 12   # frames to observe ball movement after contact
MOVEMENT_THRESHOLD_PX   =  8   # min pixel displacement to confirm contact
CONTACT_BBOX_PAD        =  4   # extra pixels on cue bbox for overlap test

# Class-vote stability
CLASS_VOTE_WINDOW = 50          # rolling window size (frames) per track ID

# ── Rail-contact foul ─────────────────────────────────────────────────────
RAIL_MARGIN_PX = 40   # tune upward if your camera angle makes rails appear
POST_SHOT_GRACE_FRAMES = 45  # extra frames after cue rests to let balls reach rails (~1.5 s @ 30 fps)

# ── Trajectory-based first-contact detection ──────────────────────────────────
TRAJECTORY_PROXIMITY_PX  = 35   # max px from the cue's projected path for a ball to be a candidate
TRAJECTORY_LOOKAHEAD_PX  = 400  # how far ahead (px) to extend the trajectory line


# ═══════════════════════════ Geometry helpers ═════════════════════════════

def bbox_center(b):
    return ((b[0] + b[2]) * 0.5, (b[1] + b[3]) * 0.5)

def pt_dist(a, b):
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5

def boxes_overlap(b1, b2, pad=0):
    """xyxy overlap; optional symmetric pixel padding applied to b1."""
    a = (b1[0] - pad, b1[1] - pad, b1[2] + pad, b1[3] + pad)
    return not (a[2] < b2[0] or b2[2] < a[0] or
                a[3] < b2[1] or b2[3] < a[1])

#NEW CHANGES

def pt_to_line_segment_dist(p, v, w):
    """
    Returns the shortest distance from point p to the line segment vw.
    p = object ball center
    v = previous cue center
    w = current cue center
    """
    px, py = p
    vx, vy = v
    wx, wy = w
    
    # Length of the line segment squared
    l2 = (wx - vx)**2 + (wy - vy)**2
    if l2 == 0.0:
        return pt_dist(p, v) # Cue didn't move
        
    # Change (wy - wy) to (wy - vy)
    t = max(0, min(1, ((px - vx) * (wx - vx) + (py - vy) * (wy - vy)) / l2))
    
    proj_x = vx + t * (wx - vx)
    proj_y = vy + t * (wy - vy)
    
    return pt_dist(p, (proj_x, proj_y))


# ═══════════════════════════ Data classes ════════════════════════════════

@dataclasses.dataclass
class Track:
    track_id:   int
    bbox:       list    
    stable_cls: int     
    center:     tuple   


@dataclasses.dataclass
class PendingContact:
    contacted_track_id: int
    contacted_class:    int
    pos_at_contact:     tuple          
    created_frame:      int
    shot_target_class:  Optional[int]  


# ═══════════════════════════ Application ═════════════════════════════════

class FoulDetectorApp:

    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("9-Ball Foul Detector — ByteTrack")
        self.root.geometry("1280x860")
        self.root.configure(bg="#12121f")

        # Resources set by the user
        self.model:      Optional[YOLO] = None
        self.model_path: str            = DEFAULT_MODEL_PATH
        self.video_path: Optional[str]  = None
        self.output_dir: Optional[str]  = None

        # I/O handles (live while processing)
        self.cap:          Optional[cv2.VideoCapture] = None
        self.video_writer: Optional[cv2.VideoWriter]  = None
        self.foul_log                                 = None  
        self.debug_log                                = None  # NEW: Frame-by-frame debug log                            = None  

        # Playback state
        self.is_playing  = False
        self.is_paused   = False
        self.fps         = 30.0
        self.frame_idx   = 0
        self._foul_count = 0

        # Per-track histories
        self.class_history: defaultdict = defaultdict(
            lambda: deque(maxlen=CLASS_VOTE_WINDOW))
        self.ball_positions: defaultdict = defaultdict(
            lambda: deque(maxlen=90))

        # Shot / foul state
        self.prev_cue_center:        Optional[tuple] = None
        self.cue_velocity:           float           = 0.0
        self.shot_in_progress:       bool            = False
        self.lowest_at_shot_start:   Optional[int]   = None
        self.is_break_shot:          bool            = False
        self.rack_frames_since_seen: int             = 999
        self.checked_contact_tids:   set             = set()
        self._low_vel_run:           int             = 0
        self.pending_contacts:       List[PendingContact]   = []
        
        # Rail & Logging requirements
        self.ball_start_info: dict = {}
        self.balls_left_rail_this_shot: set = set()
        self.contact_occurred_this_shot: bool = False
        self.post_contact_rail_hit:      bool = False
        self.post_contact_pocketed:      bool = False
        self.active_ball_ids_at_contact: dict = {}
        
        # Avoid spamming the log for continuous states
        self.event_log: deque            = deque(maxlen=6)
        self.balls_hit_rail_this_shot: set = set()
        self.potted_balls: set           = set()

        # Continuous-foul state
        self.cue_missing_frames:   int  = 0
        self.cue_in_pocket_frames: int  = 0
        self._scratch_fired:       bool = False
        self._missing_fired:       bool = False

        # Pocket-disappear state
        self.cue_was_near_pocket:          bool = False
        self.cue_near_pocket_streak:       int  = 0   # persistence counter for pocket proximity
        self.cue_gone_after_pocket_frames: int  = 0
        self._pocket_disappear_fired:      bool = False
        self.post_shot_eval_frame:         Optional[int] = None  # deferred rail evaluation

        # Table boundary estimate
        self.table_bounds: Optional[tuple] = None

        # On-screen banners
        self.foul_banner_text:        str = ""
        self.foul_banner_until_frame: int = 0

        self._build_ui()
        self._load_model_async(self.model_path)

    # ─────────────────────────────── UI ──────────────────────────────────

    def _build_ui(self):
        bar = tk.Frame(self.root, bg="#1c1c35", pady=7)
        bar.pack(side=tk.TOP, fill=tk.X)

        file_kw = dict(bg="#1f3a6e", fg="white", activebackground="#2a4f96",
                       relief="flat", padx=14, pady=7,
                       font=("Segoe UI", 10, "bold"), cursor="hand2")
        ctrl_kw = dict(bg="#2b2b4a", fg="white", activebackground="#3d3d6b",
                       relief="flat", padx=14, pady=7,
                       font=("Segoe UI", 10), cursor="hand2")

        tk.Button(bar, text="📂  Upload Video",
                  command=self.upload_video, **file_kw).pack(side=tk.LEFT, padx=5)
        tk.Button(bar, text="📁  Output Folder",
                  command=self.select_output, **file_kw).pack(side=tk.LEFT, padx=5)
        tk.Button(bar, text="🤖  Load Model",
                  command=self.load_model_dialog, **file_kw).pack(side=tk.LEFT, padx=5)

        tk.Frame(bar, bg="#555", width=2, height=36).pack(
            side=tk.LEFT, padx=14, fill=tk.Y)

        self.start_btn = tk.Button(bar, text="▶  Start",
                                   command=self.start_processing,
                                   state=tk.DISABLED, **ctrl_kw)
        self.start_btn.pack(side=tk.LEFT, padx=4)
        self.pause_btn = tk.Button(bar, text="⏸  Pause",
                                   command=self.toggle_pause,
                                   state=tk.DISABLED, **ctrl_kw)
        self.pause_btn.pack(side=tk.LEFT, padx=4)
        self.stop_btn  = tk.Button(bar, text="⏹  Stop",
                                   command=self.stop_processing,
                                   state=tk.DISABLED, **ctrl_kw)
        self.stop_btn.pack(side=tk.LEFT, padx=4)

        self.status_lbl = tk.Label(bar, text="Loading model…",
                                   fg="#7ecfff", bg="#1c1c35",
                                   font=("Segoe UI", 10))
        self.status_lbl.pack(side=tk.RIGHT, padx=14)

        self.video_lbl = tk.Label(self.root, bg="black")
        self.video_lbl.pack(fill=tk.BOTH, expand=True, padx=8, pady=6)

        log_frame = tk.Frame(self.root, bg="#12121f", pady=4)
        log_frame.pack(side=tk.BOTTOM, fill=tk.X, padx=8)
        tk.Label(log_frame, text="Foul Log", fg="white", bg="#12121f",
                 font=("Segoe UI", 11, "bold")).pack(anchor=tk.W)
        self.foul_listbox = tk.Listbox(
            log_frame, height=5,
            bg="#0d0d1f", fg="#ff7070",
            font=("Consolas", 10),
            selectbackground="#2b2b4a",
            highlightthickness=0, borderwidth=0,
        )
        self.foul_listbox.pack(fill=tk.X, pady=2)

    # ─────────────────────── Model / file loading ─────────────────────────

    def _load_model_async(self, path: str):
        def _do():
            try:
                m = YOLO(path)
                self.model      = m
                self.model_path = path
                self.root.after(0, lambda: self.status_lbl.config(
                    text=f"✔  {os.path.basename(path)}", fg="#7ecfff"))
                self.root.after(0, self._check_ready)
            except Exception as exc:
                msg = str(exc)
                self.root.after(0, lambda: self.status_lbl.config(
                    text=f"Model error: {msg}", fg="#ff7070"))
        threading.Thread(target=_do, daemon=True).start()

    def load_model_dialog(self):
        path = filedialog.askopenfilename(
            title="Select YOLO weights (.pt)",
            filetypes=[("PyTorch weights", "*.pt"), ("All files", "*.*")],
        )
        if path:
            self.status_lbl.config(text="Loading…", fg="#7ecfff")
            self._load_model_async(path)

    def upload_video(self):
        path = filedialog.askopenfilename(
            title="Select video",
            filetypes=[
                ("Video files", "*.mp4 *.avi *.mov *.mkv *.m4v"),
                ("All files", "*.*"),
            ],
        )
        if path:
            self.video_path = path
            self.status_lbl.config(
                text=f"Video: {os.path.basename(path)}", fg="white")
            self._check_ready()

    def select_output(self):
        path = filedialog.askdirectory(title="Select output folder")
        if path:
            self.output_dir = path
            self._check_ready()

    def _check_ready(self):
        if self.video_path and self.output_dir and self.model is not None:
            self.start_btn.config(state=tk.NORMAL)

    # ─────────────────────── Processing lifecycle ─────────────────────────

    def start_processing(self):
        if not (self.video_path and self.output_dir and self.model):
            messagebox.showerror("Not ready",
                                 "Load a model, choose a video, and pick an "
                                 "output folder before starting.")
            return

        self.cap = cv2.VideoCapture(self.video_path)
        if not self.cap.isOpened():
            messagebox.showerror("Error", "Could not open video file.")
            return

        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 30.0
        W = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        ts     = datetime.now().strftime("%Y%m%d_%H%M%S")
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        self.video_writer = cv2.VideoWriter(
            os.path.join(self.output_dir, f"annotated_{ts}.mp4"),
            fourcc, self.fps, (W, H),
        )
        self.foul_log = open(
            os.path.join(self.output_dir, f"foul_log_{ts}.txt"),
            "w", encoding="utf-8",
        )
        self.foul_log.write(
            f"Foul Log — {ts}\nSource: {self.video_path}\n\n")
        
        self.debug_log = open(
            os.path.join(self.output_dir, f"debug_positions_{ts}.txt"),
            "w", encoding="utf-8",
        )
        self.debug_log.write(
            f"Ball Position Debug Log — {ts}\nSource: {self.video_path}\n\n")

        self._reset_state()

        self.is_playing = True
        self.is_paused  = False
        self.start_btn.config(state=tk.DISABLED)
        self.pause_btn.config(state=tk.NORMAL, text="⏸  Pause")
        self.stop_btn.config(state=tk.NORMAL)

        threading.Thread(target=self._process_loop, daemon=True).start()

    def _reset_state(self):
        self.frame_idx               = 0
        self._foul_count             = 0
        self.class_history           = defaultdict(
            lambda: deque(maxlen=CLASS_VOTE_WINDOW))
        self.ball_positions          = defaultdict(lambda: deque(maxlen=90))
        self.prev_cue_center         = None
        self.cue_velocity            = 0.0
        self.shot_in_progress        = False
        self.lowest_at_shot_start    = None
        self.is_break_shot           = False
        self.rack_frames_since_seen  = 999
        self.checked_contact_tids    = set()
        self.balls_near_pocket           = {}
        self.disappeared_near_pocket_frames = defaultdict(int)
        self._low_vel_run            = 0
        self.pending_contacts        = []
        self.stable_pockets: list = []       
        self.pocket_candidates: dict = {}    
        self.balls_moved_this_shot: dict = {}
        self.trajectory_candidates: set  = set()   # ball IDs in/near cue path this shot
        self.first_trajectory_hit_fired: bool = False
        
        # Rail requirements & logs variables
        self.ball_start_info = {}
        self.balls_left_rail_this_shot.clear()
        self.contact_occurred_this_shot  = False
        self.post_contact_rail_hit       = False
        self.post_contact_pocketed       = False
        self.active_ball_ids_at_contact  = {}
        self.event_log.clear()
        self.balls_hit_rail_this_shot.clear()
        self.potted_balls.clear()

        self.cue_missing_frames      = 0
        self.cue_in_pocket_frames    = 0
        self._scratch_fired          = False
        self._missing_fired          = False
        self.cue_was_near_pocket          = False
        self.cue_near_pocket_streak       = 0
        self.cue_gone_after_pocket_frames = 0
        self._pocket_disappear_fired      = False
        self.post_shot_eval_frame         = None
        self.table_bounds                 = None   
        self.foul_banner_text             = ""
        self.foul_banner_until_frame      = 0
        self.foul_listbox.delete(0, tk.END)

    def toggle_pause(self):
        self.is_paused = not self.is_paused
        self.pause_btn.config(
            text="▶  Resume" if self.is_paused else "⏸  Pause")

    def stop_processing(self):
        self.is_playing = False

    def _cleanup(self):
        if self.cap:
            self.cap.release()
            self.cap = None
        if self.video_writer:
            self.video_writer.release()
            self.video_writer = None
        if self.foul_log:
            self.foul_log.close()
            self.foul_log = None
        if self.debug_log:
            self.debug_log.close()
            self.debug_log = None

        ready = bool(self.video_path and self.output_dir and self.model)
        self.start_btn.config(state=tk.NORMAL if ready else tk.DISABLED)
        self.pause_btn.config(state=tk.DISABLED, text="⏸  Pause")
        self.stop_btn.config(state=tk.DISABLED)

    # ─────────────────────── Rail & Log helpers ───────────────────────────

    def _update_table_bounds(self, pockets: list):
        if not pockets:
            return
            
        # ── 1. Register new pockets and update existing ones ──
        for p in pockets:
            px, py = p.center
            matched = False
            
            for i, (sx, sy) in enumerate(self.stable_pockets):
                if pt_dist((px, py), (sx, sy)) < 60:  
                    self.stable_pockets[i] = (sx * 0.95 + px * 0.05, sy * 0.95 + py * 0.05)
                    matched = True
                    break
                    
            if not matched:
                matched_candidate = None
                for (cx, cy) in list(self.pocket_candidates.keys()):
                    if pt_dist((px, py), (cx, cy)) < 60:
                        self.pocket_candidates[(cx, cy)] += 1
                        matched_candidate = (cx, cy)
                        if self.pocket_candidates[(cx, cy)] > 15:
                            self.stable_pockets.append((cx, cy))
                            del self.pocket_candidates[(cx, cy)]
                        break
                
                if not matched_candidate:
                    self.pocket_candidates[(px, py)] = 1

        # ── 2. Form the table bounds EXCLUSIVELY from stable pockets ──
        if len(self.stable_pockets) >= 4:
            xs = [sx for (sx, sy) in self.stable_pockets]
            ys = [sy for (sx, sy) in self.stable_pockets]
            
            self.table_bounds = (min(xs), min(ys), max(xs), max(ys))

    def _ball_at_rail(self, bbox: list, frame_shape: tuple) -> bool:
        h, w = frame_shape[:2]
        if self.table_bounds is not None:
            rx1, ry1, rx2, ry2 = self.table_bounds
        else:
            rx1, ry1, rx2, ry2 = 0, 0, w, h

        bx1, by1, bx2, by2 = bbox
        return (
            bx1 <= rx1 + RAIL_MARGIN_PX or   
            bx2 >= rx2 - RAIL_MARGIN_PX or   
            by1 <= ry1 + RAIL_MARGIN_PX or   
            by2 >= ry2 - RAIL_MARGIN_PX       
        )
        
    def _add_event_log(self, msg: str):
        """Pushes a timed event string to the top-left visual log."""
        t      = self.frame_idx / max(self.fps, 1e-6)
        ts_str = f"[{int(t // 60):02d}:{int(t % 60):02d}]"
        self.event_log.append(f"{ts_str} {msg}")

    def _evaluate_rail_checks(self):
        if not self.contact_occurred_this_shot:
            return

        # If any ball was seen overlapping a pocket, it is potted -> exempt
        if self.post_contact_pocketed:
            return

        # If a rail was hit by any ball -> exempt
        if self.post_contact_rail_hit:
            return

        # Otherwise -> FOUL
        self._register_foul("No rail contact: Neither the cue nor object balls reached a rail or pocket after contact")

    # ─────────────────────── Main processing loop ─────────────────────────

    def _process_loop(self):
        scratch_frames          = max(1, int(SCRATCH_REST_SECS    * self.fps))
        missing_thresh          = max(1, int(CUE_MISSING_SECS     * self.fps))
        shot_end_frames         = max(1, int(SHOT_END_SECS        * self.fps))
        pocket_disappear_thresh = max(1, int(POCKET_DISAPPEAR_SECS * self.fps))

        try:
            while self.is_playing and self.cap and self.cap.isOpened():
                if self.is_paused:
                    time.sleep(0.04)
                    continue

                ret, frame = self.cap.read()
                if not ret:
                    break
                self.frame_idx += 1

                # ── 1. Detect + Track ──
                try:
                    yolo_out = self.model.predict(
                        frame, verbose=False,
                        conf=CONF_THRESHOLD,
                    )[0]
                except Exception as exc:
                    print(f"[frame {self.frame_idx}] inference error: {exc}")
                    continue

                # ── 2. Parse tracks ───────────────────────────────────────
                balls:      List[Track] = []
                cue:        Optional[Track] = None
                pockets:    List[Track] = []
                cue_sticks: List[Track] = []
                
                rack_detected_this_frame = False

                if len(yolo_out.boxes) > 0:
                    # Sort boxes by confidence so we process the most confident detections first
                    sorted_boxes = sorted(yolo_out.boxes, key=lambda b: b.conf.item(), reverse=True)
                    seen_ball_classes = set()

                    for i, box in enumerate(sorted_boxes):
                        raw_cls = int(box.cls.item())
                        
                        # --- THE FIX: Use the unique ball class as the permanent ID ---
                        if raw_cls in BALL_CLASSES or raw_cls == CUE_CLASS:
                            if raw_cls in seen_ball_classes:
                                continue # Skip lower-confidence duplicate detections (ghost balls)
                            seen_ball_classes.add(raw_cls)
                            tid = raw_cls # e.g., the 9-ball will ALWAYS be ID 8.
                        else:
                            tid = 1000 + i # Give pockets and sticks arbitrary IDs so they don't clash
                        # --------------------------------------------------------------

                        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().tolist()
                        raw_cls = int(box.cls.item())
                        bbox    = [x1, y1, x2, y2]
                        center  = bbox_center(bbox)

                        stable_cls = raw_cls
                        self.ball_positions[tid].append(center)

                        t = Track(track_id=tid, bbox=bbox,
                                  stable_cls=stable_cls, center=center)

                        if stable_cls in BALL_CLASSES:
                            balls.append(t)
                        elif stable_cls == CUE_CLASS:
                            if cue is None:
                                cue = t
                        elif stable_cls == POCKET_CLASS:
                            pockets.append(t)
                        elif stable_cls == 10:
                            cue_sticks.append(t)
                        elif stable_cls == RACK_CLASS:
                            rack_detected_this_frame = True
                            
                # Update rack visibility history (30-frame rolling window protects against brief detection flickers)
                if rack_detected_this_frame:
                    self.rack_frames_since_seen = 0
                else:
                    self.rack_frames_since_seen += 1

                lowest_ball = min((b.stable_cls for b in balls), default=None)
                ball_by_id  = {b.track_id: b for b in balls}

                # --- DEBUG LOGGING ---
                if self.debug_log:
                    self.debug_log.write(f"--- Frame {self.frame_idx} ---\n")
                    if self.table_bounds:
                        rx1, ry1, rx2, ry2 = self.table_bounds
                        self.debug_log.write(f"  Table Bounds: ({rx1:.1f}, {ry1:.1f}) to ({rx2:.1f}, {ry2:.1f})\n")
                    else:
                        self.debug_log.write("  Table Bounds: None\n")
                        
                    for b in balls:
                        # FILTER: Only log the 7-ball (class 6) and 9-ball (class 8)
                        if b.stable_cls in (6, 8):
                            ball_name = "7-ball" if b.stable_cls == 6 else "9-ball"
                            self.debug_log.write(f"  {ball_name:<6} (ID: {b.track_id:3d}) | Center: ({b.center[0]:6.1f}, {b.center[1]:6.1f}) | BBox: {b.bbox}\n")
                    
                    if cue:
                        self.debug_log.write(f"  CUE    (ID: {cue.track_id:3d}) | Center: ({cue.center[0]:6.1f}, {cue.center[1]:6.1f}) | BBox: {cue.bbox}\n")
                    self.debug_log.write("\n")
                # ---------------------

                # ── 3a. Update table boundary estimate from pockets ───────

                # ── 3a. Update table boundary estimate from pockets ───────
                self._update_table_bounds(pockets)
                
                # ── 3b. Global Pocketing Check (every frame) ──────────────
                current_ball_ids = {b.track_id for b in balls}

                for b in balls:
                    if b.track_id not in self.potted_balls:
                        if any(boxes_overlap(b.bbox, p.bbox, pad=10) for p in pockets):
                            self.balls_near_pocket[b.track_id] = b.stable_cls
                        else:
                            self.balls_near_pocket.pop(b.track_id, None)
                            self.disappeared_near_pocket_frames.pop(b.track_id, None)

                for tid, cls in list(self.balls_near_pocket.items()):
                    if tid not in current_ball_ids:
                        self.disappeared_near_pocket_frames[tid] += 1
                        
                        if self.disappeared_near_pocket_frames[tid] > 5:
                            if tid not in self.potted_balls:
                                self.potted_balls.add(tid)
                                self.post_contact_pocketed = True
                                name = CLASS_NAMES.get(cls, "ball")
                                self._add_event_log(f"{name} potted")
                            
                            self.balls_near_pocket.pop(tid, None)
                            self.disappeared_near_pocket_frames.pop(tid, None)
                    else:
                        self.disappeared_near_pocket_frames[tid] = 0

                # ── 3c. Global Rail Check ─────────────────────────────────
                all_moving = balls + ([cue] if cue else [])
                
                # 1. Track if balls that started on the rail have explicitly left it
                for b in all_moving:
                    start_info = self.ball_start_info.get(b.track_id)
                    if start_info and start_info["on_rail"]:
                        if b.track_id not in self.balls_left_rail_this_shot:
                            # To leave the rail, it must clearly exit the padded boundary OR move > 40px
                            left_zone = False
                            if self.table_bounds is not None:
                                rx1, ry1, rx2, ry2 = self.table_bounds
                                bx1, by1, bx2, by2 = b.bbox
                                m = RAIL_MARGIN_PX + 10
                                left_zone = (bx1 > rx1 + m and bx2 < rx2 - m and 
                                             by1 > ry1 + m and by2 < ry2 - m)
                                             
                            dist_moved = pt_dist(b.center, start_info["center"])
                            if left_zone or dist_moved > 40:
                                self.balls_left_rail_this_shot.add(b.track_id)

                # 2. Check for new rail hits whenever contact has occurred
                if self.contact_occurred_this_shot:
                    for b in all_moving:
                        if b.track_id not in self.balls_hit_rail_this_shot:
                            if self._ball_at_rail(b.bbox, frame.shape):
                                
                                start_center = None
                                started_on_rail = False
                                
                                # Resolve physical start location (handles ID tracking switches)
                                if b.track_id in self.ball_start_info:
                                    start_center = self.ball_start_info[b.track_id]["center"]
                                    started_on_rail = self.ball_start_info[b.track_id]["on_rail"]
                                elif self.ball_start_info:
                                    # ID changed mid-shot -> find nearest starting ball
                                    _, nearest_info = min(
                                        self.ball_start_info.items(),
                                        key=lambda item: pt_dist(b.center, item[1]["center"])
                                    )
                                    # If within 40px, we assume it's the same physical ball
                                    if pt_dist(b.center, nearest_info["center"]) < 40:
                                        start_center = nearest_info["center"]
                                        started_on_rail = nearest_info["on_rail"]

                                valid_rail_hit = False
                                
                                if start_center is None:
                                    # 🚨 FIX: The ball was likely missed by YOLO on the exact frame the 
                                    # shot started. Instead of assuming it flew into the rail, we heal 
                                    # the snapshot by registering its current position as its start location.
                                    self.ball_start_info[b.track_id] = {
                                        "center": b.center,
                                        "on_rail": True
                                    }
                                else:
                                    dist_moved = pt_dist(b.center, start_center)
                                    
                                    # MUST have physically moved >= 15 pixels.
                                    if dist_moved >= 60:
                                        if started_on_rail:
                                            # If it started on the rail, require strict proof it left and returned
                                            if b.track_id in self.balls_left_rail_this_shot or dist_moved > 60:
                                                valid_rail_hit = True
                                        else:
                                            # Started OFF rail, moved >15px, and is now ON rail
                                            valid_rail_hit = True

                                if valid_rail_hit:
                                    self.balls_hit_rail_this_shot.add(b.track_id)
                                    self.post_contact_rail_hit = True
                                    name = CLASS_NAMES.get(b.stable_cls, "ball")
                                    self._add_event_log(f"{name} hit rail")

                # ── 4. Foul logic ─────────────────────────────────────────

                if cue is not None:
                    self.cue_missing_frames = 0
                    self._missing_fired     = False
                    cc = cue.center

                    if self.prev_cue_center is not None:
                        self.cue_velocity = pt_dist(cc, self.prev_cue_center)
                    self.prev_cue_center = cc

                    # Shot start
                    if (not self.shot_in_progress
                            and self.cue_velocity > SHOT_START_VEL):
                        self.shot_in_progress     = True
                        self.lowest_at_shot_start = lowest_ball
                        self.checked_contact_tids.clear()
                        self._low_vel_run         = 0
                        self.balls_hit_rail_this_shot.clear()
                        self.post_shot_eval_frame = None
                        self.balls_moved_this_shot.clear()
                        self.trajectory_candidates.clear()
                        self.first_trajectory_hit_fired = False
                        
                        _all_start_balls = balls + ([cue] if cue else [])
                        self.ball_start_info = {
                            _b.track_id: {
                                "center": _b.center,
                                "on_rail": self._ball_at_rail(_b.bbox, frame.shape),
                                "cls": _b.stable_cls  # <--- NEW: Save the class
                            } for _b in _all_start_balls
                        }
                        self.balls_left_rail_this_shot.clear()
                        
                        self.is_break_shot = (self.rack_frames_since_seen < 30)
                        
                        log_msg = "Shot started (Break Shot)" if self.is_break_shot else "Shot started"
                        self._add_event_log(log_msg)

                    # First overlap OR trajectory intersection → queue a PendingContact
                    if self.shot_in_progress and not self.contact_occurred_this_shot:
                        for ball in balls:
                            if ball.track_id not in self.checked_contact_tids:
                                
                                is_hit = False
                                
                                # Check 1: Standard Bounding Box overlap
                                if boxes_overlap(cue.bbox, ball.bbox, pad=CONTACT_BBOX_PAD):
                                    is_hit = True
                                    
                                # Check 2: Trajectory Swept-Path (catches high-speed "tunneling")
                                elif self.prev_cue_center is not None and self.cue_velocity > 5.0:
                                    # Calculate distance from object ball to the cue's path this frame
                                    path_dist = pt_to_line_segment_dist(ball.center, self.prev_cue_center, cue.center)
                                    
                                    # If the path passed within ~20 pixels (roughly a ball's radius + padding),
                                    # the cue ball swept through this space.
                                    if path_dist < 20.0: 
                                        is_hit = True
                                
                                # If either check passed, queue the ball to see if it actually moves
                                if is_hit:
                                    self.checked_contact_tids.add(ball.track_id)
                                    self.pending_contacts.append(PendingContact(
                                        contacted_track_id = ball.track_id,
                                        contacted_class    = ball.stable_cls,
                                        pos_at_contact     = ball.center,
                                        created_frame      = self.frame_idx,
                                        shot_target_class  = self.lowest_at_shot_start,
                                    ))
                                
                    # ── NEW: Track the exact frame each object ball first moves ──
                    if self.shot_in_progress:
                        for b in balls: # 'balls' only contains object balls, not the cue
                            if b.track_id not in self.balls_moved_this_shot:
                                start_info = self.ball_start_info.get(b.track_id)
                                if start_info:
                                    dist = pt_dist(b.center, start_info["center"])
                                    if dist >= MOVEMENT_THRESHOLD_PX:
                                        self.balls_moved_this_shot[b.track_id] = self.frame_idx

                    # ── TRAJECTORY RULE: First ball in cue's path to move = first hit ─────
                    # Step 1 — accumulate candidates: any ball that lies within
                    # TRAJECTORY_PROXIMITY_PX of the cue's projected travel line.
                    # Once added, a ball stays a candidate for the rest of the shot
                    # ("is OR was in the trajectory").
                    cue_hist = self.ball_positions[cue.track_id]
                    lookback = max(1, min(int(self.fps * 0.15), len(cue_hist) - 1))
                    if len(cue_hist) >= 2:
                        past_pos = cue_hist[max(0, len(cue_hist) - lookback - 1)]
                        curr_pos = cue.center
                        tdx = curr_pos[0] - past_pos[0]
                        tdy = curr_pos[1] - past_pos[1]
                        tspeed = (tdx**2 + tdy**2) ** 0.5
                        if tspeed > 3.0:
                            # Project the path forward from the cue's current position
                            extend = TRAJECTORY_LOOKAHEAD_PX / tspeed
                            far_x  = curr_pos[0] + tdx * extend
                            far_y  = curr_pos[1] + tdy * extend
                            for _b in balls:
                                if _b.track_id not in self.trajectory_candidates:
                                    _d = pt_to_line_segment_dist(
                                        _b.center, curr_pos, (far_x, far_y))
                                    if _d < TRAJECTORY_PROXIMITY_PX:
                                        self.trajectory_candidates.add(_b.track_id)

                    # Step 2 — among accumulated candidates, whoever moved first
                    # (earliest frame recorded in balls_moved_this_shot) is the
                    # ball the cue hit first.
                    if (not self.contact_occurred_this_shot
                            and not self.first_trajectory_hit_fired
                            and self.trajectory_candidates):
                        moved_candidates = {
                            tid: frm
                            for tid, frm in self.balls_moved_this_shot.items()
                            if tid in self.trajectory_candidates
                        }
                        if moved_candidates:
                            first_hit_tid = min(
                                moved_candidates, key=moved_candidates.get)
                            first_hit_cls = self.ball_start_info.get(
                                first_hit_tid, {}).get("cls")
                            if first_hit_cls is not None:
                                self.first_trajectory_hit_fired  = True
                                self.contact_occurred_this_shot  = True
                                self.post_contact_rail_hit        = False
                                self.post_contact_pocketed        = False
                                self.active_ball_ids_at_contact  = {
                                    _b.track_id: _b.stable_cls for _b in balls}
                                hit_name = CLASS_NAMES.get(
                                    first_hit_cls, f"#{first_hit_tid}")
                                self._add_event_log(
                                    f"Traj 1st hit: {hit_name}")
                                if not self.is_break_shot:
                                    target = self.lowest_at_shot_start
                                    if (target is not None
                                            and first_hit_cls != target):
                                        self._register_foul(
                                            f"Wrong ball: hit "
                                            f"{CLASS_NAMES.get(first_hit_cls, 'ball')}"
                                            f" first (target was "
                                            f"{CLASS_NAMES.get(target, target)})"
                                        )
                    # ─────────────────────────────────────────────────────────

                    # Shot end
                    if self.shot_in_progress:
                        if self.cue_velocity < SHOT_END_VEL:
                            self._low_vel_run += 1
                            if self._low_vel_run >= shot_end_frames:
                                self.shot_in_progress = False
                                self._low_vel_run     = 0
                                self._add_event_log("Shot ended")
                                # FIX 2: Defer rail evaluation by POST_SHOT_GRACE_FRAMES so
                                # balls still rolling toward the rail are captured first.
                                self.post_shot_eval_frame = (
                                    self.frame_idx + POST_SHOT_GRACE_FRAMES)
                        else:
                            self._low_vel_run = 0

                    # Scratch
                    in_pocket = any(
                        boxes_overlap(cue.bbox, p.bbox) for p in pockets)
                    if in_pocket and self.cue_velocity < SHOT_END_VEL:
                        self.cue_in_pocket_frames += 1
                        if (self.cue_in_pocket_frames >= scratch_frames
                                and not self._scratch_fired):
                            self._register_foul(
                                "Scratch: cue ball resting in pocket")
                            self._scratch_fired = True
                    else:
                        self.cue_in_pocket_frames = 0
                        self._scratch_fired       = False

                    # FIX 3: Use a streak counter so brief pocket-detection gaps
                    # don't break the "cue was near pocket" chain.  If the cue ball
                    # overlaps a pocket this frame, arm the streak for 8 frames;
                    # otherwise count down so it expires naturally.
                    if in_pocket:
                        self.cue_near_pocket_streak = 8
                    else:
                        self.cue_near_pocket_streak = max(
                            0, self.cue_near_pocket_streak - 1)

                    self.cue_was_near_pocket          = (
                        self.cue_near_pocket_streak > 0)
                    self.cue_gone_after_pocket_frames = 0
                    self._pocket_disappear_fired      = False

                else:
                    self.cue_velocity         = 0.0
                    self.cue_in_pocket_frames = 0
                    self.cue_missing_frames  += 1

                    # FIX 3: Fallback — if pocket detection was unreliable but the
                    # cue ball's last known position was close to a stable pocket,
                    # treat it as near-pocket so the disappear-scratch can still fire.
                    if (not self.cue_was_near_pocket
                            and self.prev_cue_center is not None
                            and self.stable_pockets):
                        for _px, _py in self.stable_pockets:
                            if pt_dist(self.prev_cue_center, (_px, _py)) < 70:
                                self.cue_was_near_pocket = True
                                break

                    if self.cue_was_near_pocket:
                        self.cue_gone_after_pocket_frames += 1
                        if (self.cue_gone_after_pocket_frames >= pocket_disappear_thresh
                                and not self._pocket_disappear_fired):
                            self._register_foul(
                                "Scratch: cue ball entered pocket")
                            self._pocket_disappear_fired = True
                            self._add_event_log("Cue ball scratch")

                    if (self.cue_missing_frames >= missing_thresh
                            and not self._missing_fired):
                        self._register_foul(
                            "Ball missing — potential foul")
                        self._missing_fired = True

                # ── 4b. Deferred post-shot rail evaluation ─────────────────────────────
                if (self.post_shot_eval_frame is not None
                        and self.frame_idx >= self.post_shot_eval_frame):
                    
                    # --- NEW: First-Mover Fallback Hit Detection ---
                    if not self.contact_occurred_this_shot and self.balls_moved_this_shot:
                        # Find the ball track_id that moved on the earliest frame
                        first_moved_id = min(self.balls_moved_this_shot, key=self.balls_moved_this_shot.get)
                        
                        # Retrieve what class that ball was at the start of the shot
                        if first_moved_id in self.ball_start_info:
                            first_moved_cls = self.ball_start_info[first_moved_id]["cls"]
                            
                            self.contact_occurred_this_shot = True
                            hit_name = CLASS_NAMES.get(first_moved_cls, f"#{first_moved_id}")
                            self._add_event_log(f"Fallback hit: {hit_name} moved first")
                            
                            # Standard wrong-ball foul check
                            if not self.is_break_shot:
                                target = self.lowest_at_shot_start
                                if target is not None and first_moved_cls != target:
                                    self._register_foul(
                                        f"Wrong ball: hit "
                                        f"{CLASS_NAMES.get(first_moved_cls, 'ball')} first "
                                        f"(target was {CLASS_NAMES.get(target, target)})"
                                    )
                    # ---------------------------------------------------------------

                    self._evaluate_rail_checks()
                    self.contact_occurred_this_shot = False
                    self.is_break_shot              = False
                    self.post_shot_eval_frame       = None

                # (c) Resolve pending contacts
                still_pending: List[PendingContact] = []
                for pc in self.pending_contacts:
                    age = self.frame_idx - pc.created_frame

                    if age < MOVEMENT_CONFIRM_FRAMES:
                        still_pending.append(pc)
                        continue

                    if pc.contacted_track_id in ball_by_id:
                        cur_pos = ball_by_id[pc.contacted_track_id].center
                        moved   = pt_dist(cur_pos, pc.pos_at_contact)
                    else:
                        moved = 0.0

                    if moved >= MOVEMENT_THRESHOLD_PX:
                        if not self.contact_occurred_this_shot:
                            target   = pc.shot_target_class
                            hit_name = CLASS_NAMES[pc.contacted_class]

                            self._add_event_log(f"Cue hit {hit_name}")

                            # Wrong-ball foul (EXEMPT if it's a break shot)
                            if not self.is_break_shot:
                                if target is not None and pc.contacted_class != target:
                                    self._register_foul(
                                        f"Wrong ball: hit "
                                        f"{CLASS_NAMES[pc.contacted_class]} first "
                                        f"(target was {CLASS_NAMES.get(target, target)})"
                                    )

                            self.contact_occurred_this_shot = True
                            self.post_contact_rail_hit = False
                            self.post_contact_pocketed = False
                            self.active_ball_ids_at_contact = {
                                b.track_id: b.stable_cls for b in balls
                            }
                    else:
                        self.checked_contact_tids.discard(pc.contacted_track_id)

                self.pending_contacts = still_pending

                # ── 5. Draw & write output ────────────────────────────────
                annotated = self._draw(
                    frame, balls, cue, pockets, cue_sticks, lowest_ball)

                if self.video_writer:
                    self.video_writer.write(annotated)
                self._display_frame(annotated)

                time.sleep(max(0.0, 1.0 / self.fps - 0.015))

        finally:
            self.is_playing  = False
            n_fouls          = self._foul_count
            self.root.after(0, self._cleanup)
            self.root.after(0, lambda: messagebox.showinfo(
                "Done",
                f"Processing complete.\n{n_fouls} foul(s) logged.",
            ))

    # ─────────────────────────── Drawing ─────────────────────────────────

    def _draw(self, frame, balls: List[Track], cue: Optional[Track],
              pockets: List[Track], cue_sticks: List[Track],
              lowest_ball: Optional[int]):

        out = frame.copy()
        h, w = out.shape[:2]

        if self.table_bounds is not None:
            rx1, ry1, rx2, ry2 = (int(v) for v in self.table_bounds)
            cv2.rectangle(out, (rx1, ry1), (rx2, ry2), (0, 160, 160), 1)
            m = RAIL_MARGIN_PX
            cv2.rectangle(out,
                          (rx1 + m, ry1 + m),
                          (rx2 - m, ry2 - m),
                          (0, 100, 100), 1)
            cv2.putText(out, "rail zone", (rx1 + 4, ry1 + 16),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 160, 160), 1)

        needs_rail = (self.contact_occurred_this_shot and 
                      not self.post_contact_rail_hit and 
                      not self.post_contact_pocketed)

        for p in pockets:
            x1, y1, x2, y2 = map(int, p.bbox)
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 180), 2)
            cv2.putText(out, "pocket", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 180), 1)

        for s in cue_sticks:
            x1, y1, x2, y2 = map(int, s.bbox)
            cv2.rectangle(out, (x1, y1), (x2, y2), (0, 140, 255), 2)

        for b in balls:
            x1, y1, x2, y2 = map(int, b.bbox)
            is_target       = (b.stable_cls == lowest_ball)
            is_waiting_rail = (needs_rail and is_target)

            if is_waiting_rail:
                color, thick = (0, 140, 255), 3
            elif is_target:
                color, thick = (0, 255, 60), 3
            else:
                color, thick = (0, 215, 215), 2

            cv2.rectangle(out, (x1, y1), (x2, y2), color, thick)
            
            label = CLASS_NAMES.get(b.stable_cls, f"#{b.track_id}")
            if is_waiting_rail:
                label += " ⚠ rail?"
            label += f"  #{b.track_id}"
            
            cv2.putText(out, label, (x1, max(0, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1)

        if cue is not None:
            x1, y1, x2, y2 = map(int, cue.bbox)
            cv2.rectangle(out, (x1, y1), (x2, y2), (255, 255, 255), 2)
            cv2.putText(out, f"cue  #{cue.track_id}",
                        (x1, max(0, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)

            # ─── NEW: Cue Ball Trajectory Prediction ─────────────────────
            history = self.ball_positions[cue.track_id]
            
            # Look back ~0.5 seconds for the past position. 
            # (Using 0.5s is usually better than 1s in billiards so the line doesn't 
            # calculate vectors from before it bounced off a rail, but you can change 
            # the 0.5 to 1.0 if you strictly want a full second ago).
            lookback_frames = int(self.fps * 0.2) 
            
            if len(history) >= 2:
                past_idx = max(0, len(history) - lookback_frames - 1)
                past_center = history[past_idx]
                curr_center = cue.center
                
                # Vector math: Where is it now minus where was it?
                dx = curr_center[0] - past_center[0]
                dy = curr_center[1] - past_center[1]
                
                # Calculate distance moved in that time window
                dist = (dx**2 + dy**2)**0.5
                
                # Only draw the trajectory if it's actually moving (prevents jitter when still)
                if dist > 5.0:
                    # Extrapolate forward. Scale determines how long the line stretches.
                    # scale = 2.0 means "draw the line twice as far as it traveled in the lookback window"
                    scale = 2.5 
                    
                    pred_x = int(curr_center[0] + dx * scale)
                    pred_y = int(curr_center[1] + dy * scale)
                    
                    # Draw the predicted trajectory line (Cyan color)
                    cv2.line(out, (int(curr_center[0]), int(curr_center[1])), 
                                  (pred_x, pred_y), (255, 255, 0), 2)
                    
                    # Draw a small red target dot at the predicted destination
                    cv2.circle(out, (pred_x, pred_y), 4, (0, 0, 255), -1)
            # ─────────────────────────────────────────────────────────────

        # ─── NEW LOGIC: Sort balls by rail status & movement ─────────────────────
        on_rail_names = []
        off_rail_names = []
        moved_15_names = []
        not_moved_15_names = []
        
        # Combine object balls and cue ball into one list
        all_table_balls = balls + ([cue] if cue else [])
        
        for b in all_table_balls:
            name = CLASS_NAMES.get(b.stable_cls, f"#{b.track_id}")
            
            # Rail status check
            if self._ball_at_rail(b.bbox, out.shape):
                on_rail_names.append(name)
            else:
                off_rail_names.append(name)
                
            # Movement check against starting snapshot
            start_info = self.ball_start_info.get(b.track_id)
            if start_info:
                dist_moved = pt_dist(b.center, start_info["center"])
                if dist_moved > 15:
                    moved_15_names.append(name)
                else:
                    not_moved_15_names.append(name)
            else:
                # If no start info exists (e.g., shot hasn't started yet)
                not_moved_15_names.append(name)
                
        off_rail_str  = f"Balls off rail (Must move 15px): {', '.join(off_rail_names)}"
        on_rail_str   = f"Balls on rail (Must move 60px): {', '.join(on_rail_names)}"
        not_moved_str = f"Balls that didn't exceed 15px: {', '.join(not_moved_15_names)}"
        moved_str     = f"Balls that exceed 15px: {', '.join(moved_15_names)}"
        # ────────────────────────────────────────────────────────────────────────

        # Adjusted panel size to fit 4 lines of diagnostic text
        panel_bottom = 260 + (len(self.event_log) * 25)

        ov = out.copy()
        cv2.rectangle(ov, (8, 8), (600, panel_bottom), (0, 0, 0), -1)
        cv2.addWeighted(ov, 0.55, out, 0.45, 0, out)
        cv2.rectangle(out, (8, 8), (600, panel_bottom), (0, 200, 200), 2)

        cv2.putText(out, "LOWEST BALL ON TABLE",
                    (22, 40), cv2.FONT_HERSHEY_SIMPLEX,
                    0.65, (160, 200, 200), 2)

        if lowest_ball is not None:
            lo_text  = CLASS_NAMES[lowest_ball].upper()
            lo_color = (0, 255, 80)
        else:
            lo_text  = "NONE DETECTED"
            lo_color = (130, 130, 130)
        cv2.putText(out, lo_text, (22, 85),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, lo_color, 4)

        # ─── Render the rail status and movement text ───────────────────────────
        cv2.putText(out, off_rail_str, (22, 125), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1)
        cv2.putText(out, on_rail_str, (22, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1)
        cv2.putText(out, not_moved_str, (22, 175), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (180, 255, 180), 1)
        cv2.putText(out, moved_str, (22, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 180, 180), 1)

        # Shifted the event logs down to Y=235 so they don't overlap
        y_offset = 235
        for event_str in self.event_log:
            cv2.putText(out, event_str, (22, y_offset), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (220, 220, 220), 2)
            y_offset += 25

        if (self.foul_banner_text
                and self.frame_idx < self.foul_banner_until_frame):
            banner = f"  FOUL: {self.foul_banner_text}  "
            (tw, th), _ = cv2.getTextSize(
                banner, cv2.FONT_HERSHEY_SIMPLEX, 0.85, 2)
            bx = max(480, w // 2 - tw // 2)
            by = 20
            cv2.rectangle(out, (bx, by),
                          (bx + tw, by + th + 20), (0, 0, 190), -1)
            cv2.rectangle(out, (bx, by),
                          (bx + tw, by + th + 20), (80, 80, 255), 2)
            cv2.putText(out, banner, (bx, by + th + 7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2)

        return out

    # ──────────────────────── Utilities ──────────────────────────────────

    def _register_foul(self, reason: str):
        t      = self.frame_idx / max(self.fps, 1e-6)
        ts_str = f"[{int(t // 60):02d}:{int(t % 60):02d}]"
        entry  = f"[{ts_str}]  {reason}"

        self._foul_count            += 1
        self.foul_banner_text        = reason
        self.foul_banner_until_frame = (
            self.frame_idx + int(FOUL_BANNER_SECS * self.fps))

        if self.foul_log:
            self.foul_log.write(entry + "\n")
            self.foul_log.flush()

        def _add():
            self.foul_listbox.insert(tk.END, entry)
            self.foul_listbox.see(tk.END)
        self.root.after(0, _add)

    def _display_frame(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        max_w = 1200
        if w > max_w:
            scale     = max_w / w
            frame_bgr = cv2.resize(frame_bgr, (max_w, int(h * scale)))
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        def _update():
            img   = Image.fromarray(rgb)
            imgtk = ImageTk.PhotoImage(image=img)
            self.video_lbl.imgtk = imgtk
            self.video_lbl.config(image=imgtk)
        self.root.after(0, _update)


# ═══════════════════════════ Entry point ══════════════════════════════════

if __name__ == "__main__":
    root = tk.Tk()
    FoulDetectorApp(root)
    root.mainloop()